# Imports & Functions

In [18]:
import pandas as pd
import numpy as np
from march_madness.config import INTERIM_DATA_DIR, PROCESSED_DATA_DIR

In [20]:
def abstract_result_columns(results, side):
    assert side in ["W", "L"], "side must be 'W' or 'L'"
    assert isinstance(results, pd.DataFrame), "results must be a DataFrame"
    
    df = results.copy()

    team_chr = "team_"
    opp_chr = "opp_"
    
    w_chr = team_chr if side == "W" else opp_chr
    l_chr = opp_chr if side == "W" else team_chr
    
    # Rename W columns (excluding WLoc)
    df.columns = [
        w_chr + col[1:] if col.startswith("W") and col != "WLoc" else col
        for col in df.columns
    ]
    
    # Rename L columns
    df.columns = [
        l_chr + col[1:] if col.startswith("L") else col
        for col in df.columns
    ]
    
    df["team_won"] = 1 if side == "W" else 0
    
    return df

# Load Data

In [19]:
df_reg_detail_results = pd.read_parquet(INTERIM_DATA_DIR / 'df_reg_detail_results.parquet', engine='pyarrow')
df_tn_detail_results = pd.read_parquet(INTERIM_DATA_DIR / 'df_tn_detail_results.parquet', engine='pyarrow')
df_seeds = pd.read_parquet(INTERIM_DATA_DIR / 'df_seeds.parquet', engine='pyarrow')

# Build feature set

In [21]:
df_reg_team_results = pd.concat([
    abstract_result_columns(df_reg_detail_results, side = 'W'),
    abstract_result_columns(df_reg_detail_results, side = 'L')
])

In [22]:
team_cols = [x for x in df_reg_team_results.columns if 'team_' in x]
opp_cols = [x for x in df_reg_team_results.columns if 'opp_' in x]

In [23]:
df_team_agg = df_reg_team_results.groupby(['Season', 'team_TeamID']).sum().reset_index()[['Season', 'team_TeamID']+team_cols]
df_opp_agg = df_reg_team_results.groupby(['Season', 'opp_TeamID']).sum().reset_index()[['Season', 'opp_TeamID']+opp_cols]

In [24]:
df_grp = df_reg_team_results.groupby(['Season', 'team_TeamID'])
df_grp = df_grp.agg(games = ('Season', 'count'),
                    last_game = ('DayNum', 'max'),
                    **{f'ttl_{col}': (col, 'sum') for col in team_cols+opp_cols}).reset_index()

In [25]:
df_grp = df_grp.drop('ttl_opp_TeamID',axis=1)
df_grp = df_grp.rename(columns={'ttl_team_won':'wins'})
df_grp['win_pct'] = df_grp['wins'] / df_grp['games']

In [26]:
df_grp['team_fgm3_percent'] = df_grp['ttl_team_FGM3'] / df_grp['ttl_team_FGA3']
df_grp['team_fgm_percent'] = df_grp['ttl_team_FGM'] / df_grp['ttl_team_FGA']
df_grp['team_ft_percent'] = df_grp['ttl_team_FTM'] / df_grp['ttl_team_FTA']

df_grp['opp_fgm3_percent'] = df_grp['ttl_opp_FGM3'] / df_grp['ttl_opp_FGA3']
df_grp['opp_fgm_percent'] = df_grp['ttl_opp_FGM'] / df_grp['ttl_opp_FGA']
df_grp['opp_ft_percent'] = df_grp['ttl_opp_FTM'] / df_grp['ttl_opp_FTA']

In [27]:
ttl_cols = [x for x in df_grp.columns if 'ttl_' in x]

for col in ttl_cols:
    pg_col_name = col.replace('ttl', 'pg')
    df_grp[pg_col_name] = df_grp[col] / df_grp['games']

In [28]:
pgf_cols = [x for x in df_grp.columns if ('pg_' in x) and (('FG' in x) or ('FT' in x))]

df_reg_final = df_grp.drop(ttl_cols, axis=1)
df_reg_final = df_reg_final.drop(pgf_cols, axis=1)
df_reg_final = df_reg_final.drop(['games', 'last_game', 'wins'], axis=1)

In [29]:
df_tn_detail_results['spread'] = df_tn_detail_results['WScore'] - df_tn_detail_results['LScore']
df_tn_detail_results = df_tn_detail_results[['Season', 'DayNum', 'NumOT', 'spread', 'WTeamID', 'LTeamID']]
df_tn_detail_results['lower_id'] = np.where(df_tn_detail_results['WTeamID'] < df_tn_detail_results['LTeamID'], df_tn_detail_results['WTeamID'], df_tn_detail_results['LTeamID'])
df_tn_detail_results['higher_id'] = np.where(df_tn_detail_results['WTeamID'] < df_tn_detail_results['LTeamID'], df_tn_detail_results['LTeamID'], df_tn_detail_results['WTeamID'])
df_tn_detail_results['lower_id_won'] = np.where(df_tn_detail_results['lower_id'] == df_tn_detail_results['WTeamID'], 1, 0)
df_tn_detail_results = df_tn_detail_results.drop(['WTeamID', 'LTeamID'], axis=1)

In [30]:
df_seeds['seed_num'] = df_seeds['Seed'].str.replace(r'[^0-9.-]', '', regex=True).astype(int)


df_tn_final = df_tn_detail_results.merge(df_reg_final, left_on = ['Season', 'lower_id'], right_on = ['Season', 'team_TeamID'])
df_tn_final = df_tn_final.merge(df_reg_final, left_on = ['Season', 'higher_id'], right_on = ['Season', 'team_TeamID'], suffixes = ['_lower', '_higher'])
df_tn_final = df_tn_final.merge(df_seeds, left_on = ['Season', 'lower_id'], right_on = ['Season', 'TeamID'])
df_tn_final = df_tn_final.merge(df_seeds, left_on = ['Season', 'higher_id'], right_on = ['Season', 'TeamID'], suffixes = ['_lower', '_higher'])

# Export features

In [31]:
df_reg_final.to_parquet(PROCESSED_DATA_DIR / 'df_reg_final.parquet', engine='pyarrow', index=False)
df_tn_final.to_parquet(PROCESSED_DATA_DIR / 'df_tn_final.parquet', engine='pyarrow', index=False)